# Lab 4 — Why It Matters: Retrieval Quality IS Answer Quality

**Thesis:** In a RAG (Retrieval-Augmented Generation) pipeline, answer quality is *bounded* by retrieval quality. Giving a better LLM bad context produces a bad answer. Giving any LLM excellent context produces an excellent answer. Most of a strong RAG pipeline is a **database problem**, not a model problem.

## What you'll learn
- How a RAG pipeline works mechanically: retrieve → build prompt → generate
- How to see the *exact* prompt sent to the LLM — what the model actually reads
- The GOOD vs BAD context experiment: same model, same question, different retrieval → different answers
- Why security and access control live at the database layer (not the LLM)
- Citation-grounded prompting: making the LLM reference its sources

## LLM access
This notebook uses the **Elastic Inference Service** to call `claude-haiku-4.5` — the same `ES_API_KEY` that authenticates your search queries also authenticates the LLM call. No separate Anthropic key needed.

> ⏱ **The first LLM call is cold (~10–20s)** — that's normal warm-up latency, not a hang. Subsequent calls are faster.

## Before you start
- **In Instruqt:** credentials are pre-configured — just run the cells.
- **Re-running from the repo:** `export ES_ENDPOINT=https://...` and `export ES_API_KEY=...`

In [ ]:
# --- Workshop helpers (inline — same block across all 4 notebooks) ---

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=60)

def show_hits(resp, fields=("id", "title", "trap_type"), score=True):
    hits = resp["hits"]["hits"]
    if not hits:
        print("  (no hits)"); return
    for rank, h in enumerate(hits, 1):
        src = h.get("_source", {})
        cols = "  ".join(str(src.get(f, "")) for f in fields)
        s = f"  {h['_score']:.4f}" if score and h.get("_score") is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

def r_semantic(query):
    return {"standard": {"query": {"semantic": {"field": "body_semantic", "query": query}}}}

def r_bm25(query):
    return {"standard": {"query": {"multi_match": {
        "query": query, "fields": ["title^3", "body"], "type": "best_fields"}}}}

def r_rrf(query, rank_constant=60, rank_window_size=100):
    return {"rrf": {"retrievers": [r_bm25(query), r_semantic(query)],
                    "rank_constant": rank_constant, "rank_window_size": rank_window_size}}

def search(retriever, size=5, source=("id","title","trap_type","version_tags")):
    return es.search(index=INDEX, retriever=retriever, size=size, source=list(source))

print("✓ Helpers loaded")

In [ ]:
# --- LLM helpers ---
# Uses the Elastic Inference Service: ES_API_KEY authenticates both search AND LLM.
# No separate Anthropic key needed.

LLM_INFERENCE_ID = ".anthropic-claude-4.5-haiku-chat_completion"

SYSTEM_PROMPT = (
    "You are a helpful Elasticsearch documentation assistant. "
    "Answer the question using ONLY the provided documentation context. "
    "If the context does not contain enough information, say so clearly."
)

def hybrid_search(query, size=5):
    """RRF hybrid retrieval — returns flat dicts WITH body text for prompt building."""
    resp = es.search(
        index=INDEX,
        retriever=r_rrf(query),
        size=size,
        source=["id", "title", "url", "body"]
    )
    return [
        {
            "id":    h["_source"]["id"],
            "title": h["_source"]["title"],
            "url":   h["_source"].get("url", ""),
            "body":  h["_source"]["body"],
            "score": h["_score"],
        }
        for h in resp["hits"]["hits"]
    ]

def build_prompt(context_docs, question):
    """Assemble a context-stuffed prompt from retrieved docs."""
    blocks = [
        f"[Source {i}: {d['title']}]\n{d['body']}"
        for i, d in enumerate(context_docs, 1)
    ]
    return "Context:\n" + "\n\n---\n\n".join(blocks) + f"\n\nQuestion: {question}"

def synthesize(context_docs, question, show_prompt=False):
    """Build prompt, call LLM via Elastic Inference Service (streaming), return answer."""
    user_msg = build_prompt(context_docs, question)   # full bodies go to the model

    if show_prompt:
        # Display-only truncation (600 chars per doc body) so the cell stays readable on screen;
        # the model still receives full text in user_msg above.
        preview_blocks = [
            f"[Source {i}: {d['title']}]\n{d['body'][:600]}{'...' if len(d['body']) > 600 else ''}"
            for i, d in enumerate(context_docs, 1)
        ]
        preview = "Context:\n" + "\n\n---\n\n".join(preview_blocks) + f"\n\nQuestion: {question}"
        print("=== PROMPT SENT TO LLM (bodies truncated to 600 chars for display) ===")
        print(preview)
        print("=== END PROMPT ===\n")

    # chat_completion task type requires streaming — collect all chunks into one string
    resp = requests.post(
        f"{ES_ENDPOINT}/_inference/chat_completion/{LLM_INFERENCE_ID}/_stream",
        headers={"Authorization": f"ApiKey {ES_API_KEY}", "Content-Type": "application/json"},
        json={"messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ]},
        stream=True,
        timeout=60,
    )
    if not resp.ok:
        print(f"LLM call failed {resp.status_code}: {resp.text}")
        resp.raise_for_status()

    chunks = []
    for line in resp.iter_lines():
        if not line:
            continue
        line = line.decode("utf-8")
        if not line.startswith("data: "):
            continue
        data_str = line[6:]
        if data_str.strip() == "[DONE]":
            break
        try:
            chunk = json.loads(data_str)
            delta = chunk.get("choices", [{}])[0].get("delta", {})
            if "content" in delta:
                chunks.append(delta["content"])
        except json.JSONDecodeError:
            pass

    return "".join(chunks)

print("\u2713 LLM helpers loaded")
print(f"  Using inference endpoint: {LLM_INFERENCE_ID}")

In [ ]:
# Connect + count sanity
info = es.info()
count = es.count(index=INDEX)["count"]
print(f"Connected to ES {info['version']['number']} | {count} docs in '{INDEX}'")

## A RAG pipeline in 3 steps

Retrieval-Augmented Generation is not a complex architecture. It's literally:

```
1. RETRIEVE   → search for relevant documents using the user's question
2. BUILD PROMPT → stuff those documents into the LLM's context window
3. GENERATE   → call the LLM with that context; it answers from the docs
```

The key insight we'll demonstrate: **step 1 determines everything**. The LLM can only answer well if step 1 gave it the right documents. A better LLM cannot compensate for wrong documents in step 2.

Let's watch each step explicitly.

> ⏱ **The first LLM call is cold (~10–20s)** — that's the inference service spinning up. Subsequent calls are faster.

## Step 1: Retrieve — watch what comes back

We'll use our Lab 3 hybrid RRF retriever. It requests `body` in `_source` because the LLM needs the full text.

In [ ]:
question = "user can't log in"

print(f"Question: {question!r}\n")
print("Retrieving top 5 documents...\n")

docs = hybrid_search(question)

for i, d in enumerate(docs, 1):
    print(f"  Source {i}: [{d['id']}] {d['title']}")
    print(f"    Score: {d['score']:.4f}")
    print(f"    Body preview: {d['body'][:200].strip()}...")
    print()
    print('-' * 60)


## Step 2: See the EXACT prompt sent to the LLM

We're not going to abstract away the prompt — you'll see exactly what the model receives.

The LLM doesn't "know" anything about Elasticsearch authentication — it only knows what we stuffed into the context window. The prompt is the complete information it works from. If the retrieved documents are wrong, the prompt is wrong, and the answer will be wrong.

`show_prompt=True` prints the assembled context before calling the model. (Bodies are truncated to 600 chars in the display; the model sees the full text.)

In [ ]:
print(f"Question: {question!r}\n")
answer = synthesize(docs, question, show_prompt=True)
print("=== LLM ANSWER ===")
print(answer)

## The experiment: GOOD vs BAD retrieval — same model, same question

We're going to run the same question twice:
1. **GOOD:** hybrid RRF retrieval → semantically relevant documents
2. **BAD:** deliberately wrong retrieval — documents about unrelated topics

The model is identical. The question is identical. The only difference is what was retrieved.

The BAD case is forced deterministically by filtering retrieval to an off-topic `trap_type` — we're not relying on luck.

In [ ]:
# ─── GOOD CONTEXT — hybrid retrieval finds genuinely relevant docs ───────────
question = "How do I configure SAML authentication in Elasticsearch?"

print("GOOD CONTEXT: hybrid RRF retrieval")
good_docs = hybrid_search(question)
print("Retrieved:")
for i, d in enumerate(good_docs, 1):
    print(f"  Source {i}: [{d['id']}] {d['title']}")

print("\nGenerating answer...")
good_answer = synthesize(good_docs, question)

print("\n=== GOOD ANSWER ===")
print(good_answer)

In [ ]:
# ─── BAD CONTEXT — retrieval forced to off-topic docs via a term filter ──────
# We filter to docs with trap_type = 'version-specific' (version release notes).
# These have ZERO relevance to SAML authentication — but we're using the same
# question and the same model. The filter guarantees the failure is deterministic.

print("BAD CONTEXT: retrieval forced to off-topic docs (version-specific trap_type)")

# Force wrong docs with a filter on trap_type
bad_retriever = {"rrf": {
    "retrievers": [
        {"standard": {"query": {"bool": {
            "must": [{"multi_match": {"query": question, "fields": ["title^3", "body"]}}],
            "filter": [{"term": {"trap_type": "version-specific"}}]}}}},
        {"standard": {"query": {"bool": {
            "must": [{"semantic": {"field": "body_semantic", "query": question}}],
            "filter": [{"term": {"trap_type": "version-specific"}}]}}}},
    ],
    "rank_constant": 60, "rank_window_size": 100,
}}

bad_resp = es.search(index=INDEX, retriever=bad_retriever, size=5,
                      source=["id", "title", "url", "body"])
bad_docs = [
    {"id": h["_source"]["id"], "title": h["_source"]["title"],
     "url": h["_source"].get("url",""), "body": h["_source"]["body"], "score": h["_score"]}
    for h in bad_resp["hits"]["hits"]
]

print("Retrieved (wrong docs):")
for i, d in enumerate(bad_docs, 1):
    print(f"  Source {i}: [{d['id']}] {d['title']}")

print("\nGenerating answer with bad context...")
bad_answer = synthesize(bad_docs, question)

print("\n=== BAD ANSWER ===")
print(bad_answer)

In [ ]:
# Side-by-side comparison
print(f"Question: {question!r}")
print("=" * 70)
print("GOOD CONTEXT ANSWER:")
print("-" * 70)
print(good_answer)
print()
print("BAD CONTEXT ANSWER:")
print("-" * 70)
print(bad_answer)
print()
print("=" * 70)
print("\n💡 The model didn't get dumber. The retrieval got worse.")
print("   This is not a model quality problem. It's a retrieval quality problem.")

## Security at the database layer — not the LLM

Notice something about the BAD context experiment: the filter (`trap_type = version-specific`) worked perfectly. The LLM **could not see** any of the SAML authentication documents — not because we told the LLM to hide them, but because Elasticsearch never returned them in the first place.

This is the architecture of secure RAG:

```
User request
    │
    ▼
Elasticsearch retrieval  ← access control evaluated HERE, before retrieval
    │                       (document-level security, RBAC, filter queries)
    │  Only docs the user is authorized to see are returned
    ▼
LLM prompt building      ← LLM only sees what ES returned
    │
    ▼
LLM generation           ← cannot hallucinate or expose unauthorized content
                            because it was never in the context
```

> **Workshop note:** In the BAD context experiment, we used a manual `trap_type` filter to force wrong documents into the retriever. We wrote that filter ourselves — that's not production DLS. We did it this way because setting up real RBAC roles takes more time than a workshop allows.
>
> In production, **Document-Level Security (DLS)** automates exactly this: you attach role-based filters to users or API keys, and Elasticsearch applies them at query time — no filter clause required in application code. A contractor's API key physically cannot retrieve documents tagged as confidential. The LLM can't expose what the retriever never returned.

**The takeaway:** Security in RAG lives at the retrieval layer, not in the LLM prompt. Prompt-level rules ("don't mention confidential documents") can be bypassed — database-level rules cannot.


## Citation-grounded prompting — making answers verifiable

We prepend each document with `[Source N: Title]`. We can tell the LLM to cite those sources in its answer — making every claim traceable back to a specific retrieved document.

This dramatically reduces hallucination risk: if the LLM can't cite a source, it shouldn't make the claim. And users can click through to verify.

In [ ]:
CITATION_SYSTEM_PROMPT = (
    "You are a helpful Elasticsearch documentation assistant. "
    "Answer the question using ONLY the provided documentation context. "
    "For each claim you make, cite the source using [Source N] notation. "
    "If the context does not contain enough information, say so and do not speculate."
)

def synthesize_with_citations(context_docs, question):
    """Synthesize with citation instruction."""
    user_msg = build_prompt(context_docs, question)
    resp = requests.post(
        f"{ES_ENDPOINT}/_inference/chat_completion/{LLM_INFERENCE_ID}",
        headers={"Authorization": f"ApiKey {ES_API_KEY}", "Content-Type": "application/json"},
        json={"messages": [
            {"role": "system", "content": CITATION_SYSTEM_PROMPT},
            {"role": "user",   "content": user_msg},
        ]},
        timeout=60,
    )
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"]

print(f"Question: {question!r}\n")
cited_answer = synthesize_with_citations(good_docs, question)
print("=== CITED ANSWER ===")
print(cited_answer)
print("\nSource list:")
for i, d in enumerate(good_docs, 1):
    print(f"  [Source {i}] {d['title']} — {d['url']}")

## Try it: Run your own question

Replace `your_question` below with any Elasticsearch question and run the cell. The hybrid retriever will find the most relevant docs in our 60-doc corpus and the LLM will answer from them.

In [ ]:
def ask(your_question):
    """One-liner: hybrid search → LLM synthesis → print answer."""
    docs = hybrid_search(your_question)
    print(f"Retrieved {len(docs)} docs:")
    for i, d in enumerate(docs, 1):
        print(f"  {i}. [{d['id']}] {d['title']} (score={d['score']:.4f})")
    print()
    answer = synthesize(docs, your_question)
    print("=== ANSWER ===")
    print(answer)

# Replace with your question:
ask("How do I monitor shard allocation in Elasticsearch?")

## Take-home: Multi-hop retrieval agent

A single-turn RAG pipeline answers one question from one retrieval. But some questions require multiple lookups — find an answer, then follow up on something in that answer, then follow up again.

The cell below implements a minimal multi-hop agent: it asks the LLM whether a follow-up retrieval is needed, and if so, what to retrieve next. The LLM uses the Inference API (same API key) to decide.

This runs entirely against your Elastic project — no extra infrastructure.

In [ ]:
AGENT_SYSTEM = (
    "You are a retrieval agent. Given a question and some context, either:\n"
    "1. Answer the question if the context is sufficient. Start your response with ANSWER:  \n"
    "2. Ask for more information if needed. Start with LOOKUP: followed by the next search query.\n"
    "\nUse LOOKUP only once — if the second context doesn't answer the question, say so."
)

def agent_turn(question, context_docs, history=""):
    """One turn of the agent loop."""
    ctx = build_prompt(context_docs, question)
    user_msg = (f"Previous context:\n{history}\n\n" if history else "") + ctx
    resp = requests.post(
        f"{ES_ENDPOINT}/_inference/chat_completion/{LLM_INFERENCE_ID}/_stream",
        headers={"Authorization": f"ApiKey {ES_API_KEY}", "Content-Type": "application/json"},
        json={"messages": [
            {"role": "system", "content": AGENT_SYSTEM},
            {"role": "user",   "content": user_msg},
        ]},
        stream=True,
        timeout=60,
    )
    resp.raise_for_status()
    chunks = []
    for line in resp.iter_lines():
        if not line: continue
        line = line.decode("utf-8")
        if not line.startswith("data: "): continue
        data_str = line[6:]
        if data_str.strip() == "[DONE]": break
        try:
            delta = json.loads(data_str).get("choices", [{}])[0].get("delta", {})
            if "content" in delta: chunks.append(delta["content"])
        except json.JSONDecodeError:
            pass
    return "".join(chunks)

def multi_hop_agent(question, max_hops=2):
    """Minimal multi-hop RAG agent."""
    print(f"Question: {question!r}\n")
    history = ""
    for hop in range(1, max_hops + 1):
        print(f"--- Hop {hop}: retrieving ---")
        docs = hybrid_search(question)
        for i, d in enumerate(docs[:3], 1):
            print(f"  {i}. {d['title']}")
        response = agent_turn(question, docs, history)
        print(f"\nAgent: {response}\n")
        if response.startswith("ANSWER:"):
            return response[7:].strip()
        elif response.startswith("LOOKUP:"):
            follow_up = response[7:].strip()
            print(f"  → Following up with: {follow_up!r}")
            history += f"\n\n[Hop {hop} retrieved context]\n" + build_prompt(docs, question)
            question = follow_up
        else:
            return response  # agent answered without prefix
    return "(max hops reached)"

# Try it:
multi_hop_agent("What are the security requirements for setting up Kibana?")